# 13. lecke - Agent emlékezet


## Beállítás

Ez a jegyzetfüzet bemutatja, hogyan építhetünk fel egy utazási foglalási ügynököt **tartós memóriával** a **Microsoft Agent Framework** (MAF) segítségével.

Megtanulod, hogy az ügynök memóriájának különböző típusai — munkamemória, rövid távú és hosszú távú memória — hogyan befolyásolják, hogy az ügynök hogyan őrzi meg és használja az információkat a beszélgetések során.

**Előfeltételek:**
- Egy Microsoft Foundry projekt egy telepített chat modellel (például `gpt-5-mini`).
- Belépve az Azure CLI-vel — futtasd az `az login` parancsot a terminálodban.
- `AZURE_AI_PROJECT_ENDPOINT` — a Microsoft Foundry projekted végpontja.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modelled neve.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Az ügynök memóriatípusai

A mesterséges intelligencia ügynökei különféle memóriatípusokat használhatnak, amelyek mindegyike más-más célt szolgál:

### Munkamemória
Maga a beszélgetési szál — az egyetlen munkamenet során váltott üzenetek. Az ügynök visszautalhat korábbi üzenetekre ugyanabban a szálban az összefüggés fenntartása érdekében. A MAF-ban ez az **`agent.create_session()`** segítségével jön létre, amely egy `AgentSession`-t ad vissza.

### Rövid távú memória
Olyan információk, amelyek megmaradnak egy feladat vagy munkamenet ideje alatt, de nem tárolódnak tartósan. Például az ügynök egy többfordulós tervezési beszélgetés során összegyűjthet tényeket, amelyeket aztán a végső útiterv elkészítéséhez használ.

### Hosszú távú memória
Preferenciák és tények, amelyek **munkamenetek között** is megőrződnek. Egy visszatérő felhasználónak nem kell megismételnie diétás korlátozásait vagy utazási stílusát. A hosszú távú memóriát általában egy külső tároló — adatbázis, fájl vagy vektoralapú index — támogatja, amely az ügynök számára eszközökön keresztül elérhető.


## Munkamemória munkamenetekkel

A memória legegyszerűbb formája a beszélgetési munkamenet. Ha ugyanazt a munkamenetet (amit az `agent.create_session()`-nel hoztál létre) adod át egymást követő `agent.run()` hívásoknak, az ügynök látja az adott beszélgetés teljes előzményét, és vissza tudja idézni a korábbi részleteket.

Készítsünk egy utazási ügynököt, és mutassuk be a munkamemóriát.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Az ügynök helyesen idézte fel a költségvetést, mert mindkét üzenet ugyanahhoz a munkamenethez tartozik. Ezt hívjuk **munkamemóriának** — ez csak a munkamenet élettartamáig létezik.

### Mi történik egy új szál létrehozásakor?

Ha **új** munkamenetet hozunk létre, az ügynök nem emlékszik az előző beszélgetésre:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Hosszú távú memória minta

Ahhoz, hogy **ülések között** megjegyezzük a felhasználói preferenciákat, szükségünk van egy tartós tárolóra, amely a beszélgetési szálon kívül él. Az ügynök ehhez a tárolóhoz **eszközökön** keresztül fér hozzá — olyan függvényeken, amelyeket meghívhat az információk mentésére és előhívására.

Lent egy egyszerű memórián belüli preferencia tárolót valósítunk meg (éles használatban ezt egy adatbázissal vagy vektor indexszel támogatnánk), és eszközként tesszük elérhetővé az ügynök számára.

### Architektúra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### 1. forgatókönyv — Első alkalommal foglaló felhasználó évfordulós utazáshoz

Sarah először látogat el. Az ügynöknek el kell tárolnia az ő preferenciáit az eszközök segítségével, és azokat felhasználva ajánlania kell hoteleket.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 2. forgatókönyv — Sarah hetek múlva visszatér

Sarah egy **teljesen új szálat** indít (egy új munkamenetet szimulálva). A munkamemória üres, de a hosszú távú preferencia-tár még mindig tartalmazza az ő adatait. Az ügynöknek elő kell vennie ezt az információt, és fel kell használnia a személyre szabott ajánlásokhoz.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Összefoglaló

Ebben a leckében háromféle ügynök memóriatípust tanultál, és azt, hogyan valósítsd meg őket a Microsoft Agent Framework segítségével:

| Memória típus | MAF mechanizmus | Élettartam |
|---|---|---|
| **Munkamemória** | `agent.create_session()` | Egyetlen beszélgetés |
| **Rövid távú** | Felhalmozott kontextus egy szálon belül | Egyetlen feladat / munkamenet |
| **Hosszú távú** | Külső tárhely, amelyhez `@tool` függvényekkel férnek hozzá | Munkamenetek között |

### Főbb tanulságok
1. **`agent.create_session()`** biztosítja a munkamemóriát — az ügynök látja a teljes beszélgetési előzményt egy munkameneten belül.
2. **Az új munkamenetek elveszítik a kontextust** — hosszú távú memória nélkül az ügynök nem emlékszik a korábbi beszélgetésekre.
3. **`@tool` függvények** hidat képeznek — lehetővé teszik az ügynök számára az adatok tartós tárolását és előhívását.
4. **A személyre szabás idővel javul** — minél több preferencia tárolódik, annál jobb lesz az ügynök ajánlása.

### Való világ példák
- **Ügyfélszolgálat**: Emlékezzen az ügyfelek előzményeire és preferenciáira
- **Személyi asszisztensek**: Tartsa meg a kontextust napokon vagy heteken át
- **Egészségügy**: Kövesse a páciensek adatait és preferenciáit
- **E-kereskedelem**: Személyre szabott vásárlás előzmények alapján

### Következő lépések
- Cseréld az in-memory szótárt adatbázisra vagy vektortárra (például Azure AI Search)
- Adj hozzá memória lejárati lehetőséget időérzékeny információkhoz
- Építs több ügynökből álló rendszereket megosztott memóriával
- Fedezd fel a Cognee notebookot tudásgráfon alapuló memóriához


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
